# DSI490 — Flood Chat · Design Decisions v1.3

Notebook คุยกับ `app.agent.FloodAgent` ตัวเดียวกับ API ไม่มีสำเนา Backend Rules ใน Notebook
รันจากบนลงล่างใน kernel ที่ติดตั้ง `dspy-agent/requirements.txt` แล้ว

- ข้อความตัวอักษรเรียก Gemini เพื่อ Extraction และอาจเรียบเรียงคำถาม/ความรู้ตามแผนของโค้ด
- เหตุด่วนใช้ template สั้น; คำนวณ priority และบันทึก outbox ก่อนขั้นเรียบเรียง
- พิกัดและปุ่มไม่เรียก Extraction; อาจมีขั้นเรียบเรียงคำถามถัดไปในเคสทั่วไป
- ใช้ SQLite ในหน่วยความจำเพื่อทดลอง; ไม่มี sender จึงยังไม่ส่งให้ Agent 2 จริง
- `agent.show_last()` ดูผลรอบล่าสุด; `agent.show_state()` ดู state
- `agent = make_agent()` เริ่มการทดลองใหม่พร้อมฐานข้อมูลจำลองใหม่
- คลังความรู้/พื้นที่ใช้เฉพาะข้อมูลที่ทีมตรวจรับแล้ว หากยังไม่มีจะตอบตามข้อจำกัดจริง

การรันข้อความและส่วน Golden Scenarios จะเรียก Gemini จริงและมีค่าใช้จ่ายตามการใช้งาน


In [1]:
import os
import sys
from pathlib import Path
import dspy
from dotenv import load_dotenv

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "dspy-agent").is_dir())
APP_ROOT = ROOT / "dspy-agent"
if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))
load_dotenv(APP_ROOT / ".env", override=False)
load_dotenv(ROOT / "lab" / "1" / ".env", override=False)  # รองรับไฟล์ key เดิมของ Notebook


True

In [2]:
api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY") or os.getenv("API_KEY")
model = os.getenv("GEMINI_MODEL", "gemini/gemini-3.1-flash-lite")
if not api_key:
    raise RuntimeError("ตั้ง GEMINI_API_KEY ใน dspy-agent/.env หรือ lab/1/.env ก่อนเรียก Gemini")
lm = dspy.LM(model=model, api_key=api_key)
dspy.configure(lm=lm)
print("model =", model)


model = gemini-3.1-flash-lite


In [3]:
import json
from app.agent import FloodAgent
from app.schemas import Hotline, SCHEMA_VERSION
from app.repository import Repository
from app.knowledge import KnowledgeBase
from app.areas import AreaResolver

# โหลด config เดียวกับ API; ห้ามสร้าง verified_at ใหม่เพื่อแสร้งว่าตรวจแล้ว
hotline_config = json.loads((APP_ROOT / "app" / "hotlines_th.json").read_text(encoding="utf-8"))
HOTLINES = [Hotline.model_validate(row) for row in hotline_config["hotlines"]]
print("schema =", SCHEMA_VERSION, "| hotline review =", hotline_config.get("status"))


def make_agent():
    # Notebook จำลองการส่งคำตอบสำเร็จ และแยกข้อมูลจากฐานข้อมูล API
    # ไม่มี sender: รายงานอยู่ใน outbox เป็น pending ไม่ส่ง network ไป Agent 2
    return FloodAgent(
        hotlines=HOTLINES,
        repository=Repository("sqlite:///:memory:"),
        knowledge=KnowledgeBase(APP_ROOT / "app" / "knowledge_th.csv"),
        area_resolver=AreaResolver(APP_ROOT / "app" / "areas_singburi.csv"),
        internal_user_id="usr_notebook_demo",
    )


schema = 1.3 | hotline review = draft-unverified


สายด่วนโหลดจาก `hotlines_th.json` ชุดเดียวกับ API สถานะการตรวจยังเป็นร่างจนกว่าทีมตรวจแหล่งทางการ
Notebook ไม่ตั้งเวลา `verified_at` เอง และไม่เปิด LINE หรือ transport ของ Agent 2


In [4]:
[(hotline.key, hotline.name, hotline.number, hotline.verified_at) for hotline in HOTLINES]


[('ddpm',
  'สายด่วน ปภ. (ยังไม่ได้ตรวจ)',
  '1784',
  datetime.datetime(2026, 10, 30, 0, 0, tzinfo=TzInfo(25200))),
 ('ems',
  'การแพทย์ฉุกเฉิน (ยังไม่ได้ตรวจ)',
  '1669',
  datetime.datetime(2026, 10, 30, 0, 0, tzinfo=TzInfo(25200)))]

In [5]:
agent = make_agent()


## 1. เหตุทั่วไป: ขอจุดเกิดเหตุและเสนอ GPS ทางเลือก ก่อนถามข้อมูลต่อ

ดูคำถามจริงจาก `output.pending_question` ทุกครั้ง ข้าม GPS ได้ และการข้ามไม่เท่ากับขอลบข้อมูล


In [6]:
output = agent(input_text="น้ำท่วมบ้านค่ะ")

In [7]:
output.output_text

'รับทราบข้อมูลที่แจ้งค่ะ\n\nจุดเกิดเหตุอยู่ที่ไหนคะ? แชร์ตำแหน่ง หรือพิมพ์หมู่บ้าน ตำบล และจุดสังเกตได้ค่ะ หากแจ้งแทนผู้อื่น กรุณาระบุตำแหน่งของจุดเกิดเหตุ ตำแหน่งจะใช้ระบุจุดเกิดเหตุและแนบรายงานให้ผู้ดูแลรายงานเพื่อประสานความช่วยเหลือ หากไม่สะดวก สามารถข้ามได้ค่ะ\n\nระบบจะใช้ข้อมูลที่คุณแจ้งเพื่อจัดทำรายงานและส่งต่อให้ผู้ดูแลรายงานเพื่อประสานความช่วยเหลือค่ะ'

In [8]:
output = agent(input_text="ตำบลทับยา อินทร์บุรีค่ะ แค่อยากแจ้งไว้ก่อน")
print(output.output_text)
print("ถามต่อ:", output.pending_question.field_names if output.pending_question else [])


รับทราบข้อมูลที่แจ้งค่ะ

ขอจำนวนผู้ได้รับผลกระทบ และหากมีเด็ก ผู้สูงอายุ ผู้พิการ หญิงตั้งครรภ์ หรือผู้ป่วยที่ต้องดูแลเป็นพิเศษ ขอประเภทและจำนวนเท่าที่ทราบค่ะ
ถามต่อ: [<IncidentFieldName.AFFECTED_PEOPLE: 'affected_people'>, <IncidentFieldName.VULNERABLE_PEOPLE: 'vulnerable_people'>]


In [9]:
agent.show_last()

👤 ตำบลทับยา อินทร์บุรีค่ะ แค่อยากแจ้งไว้ก่อน
   ✦ intents=['update_incident'] target=active_incident
     - location_area: known "ตำบลทับยา อำเภออินทร์บุรี"
     - needs: known ["info_only"]
   • location_area: initial
   • needs: initial
   ➜ actions=['ask_follow_up']


In [10]:
output = agent(input_text="ไม่ทราบค่ะ")
print(output.output_text)

รับทราบว่ายังไม่ทราบข้อมูลส่วนนี้ค่ะ

หากสะดวก พอจะประมาณระดับน้ำในจุดที่พบเห็นได้ไหมคะ เช่น เป็นระดับที่เทียบกับความสูงของอะไร หรือวัดได้ประมาณกี่เซนติเมตรค่ะ


In [11]:
output = agent(input_text="80 เซนติเมตร")
print(output.output_text)

รับทราบข้อมูลที่แจ้งค่ะ

สรุปข้อมูลที่แจ้งค่ะ
- พื้นที่: ตำบลทับยา อำเภออินทร์บุรี
- สถานการณ์: น้ำท่วมบ้านค่ะ
- ความต้องการ: แจ้งให้ทราบเท่านั้น
- ผู้ได้รับผลกระทบ: ไม่ทราบ
- ผู้ที่ต้องดูแลเป็นพิเศษ: ไม่ทราบ
- ระดับน้ำ: 80 เซนติเมตร
หากข้อมูลถูกต้อง กรุณายืนยันให้ส่งรายงาน หรือแจ้งส่วนที่ต้องการแก้ไขได้ค่ะ


In [12]:
output = agent(input_text="แก้เป็น 50 เซนติเมตรค่ะ พิมพ์ผิด")
print(output.output_text)
agent.show_last()

รับทราบการแก้ไขข้อมูลค่ะ

ข้อมูลมีการเปลี่ยนแปลงหลังสรุปครั้งก่อน กรุณาตรวจสรุปล่าสุดอีกครั้งค่ะ
สรุปข้อมูลที่แจ้งค่ะ
- พื้นที่: ตำบลทับยา อำเภออินทร์บุรี
- สถานการณ์: น้ำท่วมบ้านค่ะ
- ความต้องการ: แจ้งให้ทราบเท่านั้น
- ผู้ได้รับผลกระทบ: ไม่ทราบ
- ผู้ที่ต้องดูแลเป็นพิเศษ: ไม่ทราบ
- ระดับน้ำ: 50 เซนติเมตร
หากข้อมูลถูกต้อง กรุณายืนยันให้ส่งรายงาน หรือแจ้งส่วนที่ต้องการแก้ไขได้ค่ะ
👤 แก้เป็น 50 เซนติเมตรค่ะ พิมพ์ผิด
   ✦ intents=['correct_information', 'confirm_report'] target=active_incident
     - water_level: known {"raw": "50 เซนติเมตร", "descriptor": null, "cm": 50.0} (Gemini: แก้ไข)
   • water_level: correction
   ➜ actions=['confirm_report']
   ℹ️ ข้อมูลเปลี่ยนหลังแสดงสรุป ต้องยืนยันสรุปใหม่


In [13]:
output = agent(input_text="ถูกต้องค่ะ ส่งได้เลย")
print(output.output_text)
print(output.incident_status, output.next_actions, output.delivery_status)
# ถ้าข้อมูลเพิ่งเปลี่ยนในคำตอบก่อนหน้า ต้องตรวจสรุปล่าสุดแล้วค่อยยืนยันอีกครั้ง


บันทึกข้อมูลไว้แล้ว กำลังส่งต่อค่ะ
IncidentStatus.SUBMITTED [<NextAction.SUBMIT_REPORT: 'submit_report'>] DeliveryStatus.PENDING


In [14]:
agent.show_state()

incident inc_e3cb09e84a3a  status=submitted  matching=new  urgent=[]
  location_area          known       ถาม 1  {"raw": "ตำบลทับยา อำเภออินทร์บุรี", "stated_level": "unspecified", "match_status": "unverified", "candidates": []}
  location_landmark      not_asked   ถาม 0  
  location_coordinates   unanswered  ถาม 1  
  situation              known       ถาม 0  {"text": "น้ำท่วมบ้านค่ะ", "tags": ["water_in_house"]}
  needs                  known       ถาม 0  ["info_only"]
  affected_people        unknown     ถาม 1  
  vulnerable_people      unknown     ถาม 1  
  water_level            known       ถาม 1  {"raw": "50 เซนติเมตร", "cm": 50.0}  (เปลี่ยน 1 ครั้ง)
  contact                not_asked   ถาม 0  
  คำขอพิกัด: ['ask_location:pending']


## 2. เหตุด่วน: บันทึกและเตรียมส่งทันที แล้วถามต่อครั้งละหนึ่งเรื่อง

เบอร์ที่พิมพ์มาเป็นข้อมูลสมมติ ยังไม่เป็น consent: เก็บรอยืนยัน ไม่แนบรายงาน ไม่ส่ง Gemini
คำขออนุญาตใช้เบอร์จะแสดงเมื่อถึงขั้น contact ตามลำดับคำถาม


In [15]:
agent = make_agent()
output = agent(input_text="ช่วยด้วยค่ะ ยายติดอยู่บนหลังคา น้ำขึ้นเร็วมาก หมู่ 3 ตำบลน้ำตาล เบอร์ 081-234-5678")
print(output.output_text)
print(output.incident_status, output.next_actions, output.delivery_status)


จากข้อมูลที่แจ้ง กรุณาติดต่อช่องทางฉุกเฉินเพื่อขอความช่วยเหลือทันทีค่ะ
• สายด่วน ปภ. (ยังไม่ได้ตรวจ) 1784
• การแพทย์ฉุกเฉิน (ยังไม่ได้ตรวจ) 1669

บันทึกข้อมูลไว้แล้ว กำลังส่งต่อค่ะ

จุดเกิดเหตุอยู่ที่ไหนคะ? แชร์ตำแหน่ง หรือพิมพ์หมู่บ้าน ตำบล และจุดสังเกตได้ค่ะ หากแจ้งแทนผู้อื่น กรุณาระบุตำแหน่งของจุดเกิดเหตุ ตำแหน่งจะใช้ระบุจุดเกิดเหตุและแนบรายงานให้ผู้ดูแลรายงานเพื่อประสานความช่วยเหลือ หากไม่สะดวก สามารถข้ามได้ค่ะ

ระบบจะใช้ข้อมูลที่คุณแจ้งเพื่อจัดทำรายงานและส่งต่อให้ผู้ดูแลรายงานเพื่อประสานความช่วยเหลือค่ะ
IncidentStatus.SUBMITTED [<NextAction.SUBMIT_REPORT: 'submit_report'>, <NextAction.ASK_FOLLOW_UP: 'ask_follow_up'>] DeliveryStatus.PENDING


In [16]:
agent.show_last()

👤 ช่วยด้วยค่ะ ยายติดอยู่บนหลังคา น้ำขึ้นเร็วมาก หมู่ 3 ตำบลน้ำตาล เบอร์ [เบอร์โทร]
   ✦ intents=['report_flood', 'request_help'] target=new_incident
     - location_area: known "หมู่ 3 ตำบลน้ำตาล"
     - situation: known {"text": "น้ำขึ้นเร็วมาก", "tags": ["water_rising"]}
     - vulnerable_people: known {"present": true, "groups": [{"type": "elderly", "count": 1, "is_approximate": false}]}
     - contact: known "[เบอร์โทร]"
   • location_area: initial
   • situation: initial
   • vulnerable_people: initial
   keyword พบ: ['help_requested', 'people_trapped']
   ⚠️ flag ใหม่: ['help_requested', 'people_trapped']
   ➜ actions=['submit_report', 'ask_follow_up'] submit=urgent_flags
   ℹ️ พบเบอร์: รอยืนยันอนุญาต ไม่แนบรายงาน ไม่ส่ง Gemini


In [17]:
output = agent(input_text="อยู่หลังวัดค่ะ")
print(output.output_text)

บันทึกข้อมูลไว้แล้ว กำลังส่งต่อค่ะ

มีผู้ได้รับผลกระทบหรือไม่ หากมี ขอจำนวนเท่าที่ทราบค่ะ


In [18]:
agent.show_state()
print("เบอร์ที่ยืนยันแล้ว:", len(agent.contacts))
print("คำขออนุญาตใช้เบอร์:", [request.status for request in agent.contact_requests])
# ไม่พิมพ์ phone จริงหรือ dump ระเบียน consent request ที่ยังมีเบอร์


incident inc_9b060e0284b9  status=submitted  matching=new  urgent=['help_requested', 'people_trapped']
  location_area          known       ถาม 0  {"raw": "หมู่ 3 ตำบลน้ำตาล", "stated_level": "unspecified", "match_status": "unverified", "candidates": []}
  location_landmark      known       ถาม 0  "หลังวัด"
  location_coordinates   unanswered  ถาม 1  
  situation              known       ถาม 0  {"text": "ช่วยด้วยค่ะ ยายติดอยู่บนหลังคา น้ำขึ้นเร็วมาก หมู่ 3 ตำบลน้ำตาล เบอร์ [เบอร์โทร]", "tags": ["water_rising"]}
  needs                  not_asked   ถาม 0  
  affected_people        not_asked   ถาม 1  
  vulnerable_people      known       ถาม 0  {"present": true, "groups": [{"type": "elderly", "count": 1, "is_approximate": false}]}
  water_level            not_asked   ถาม 0  
  contact                not_asked   ถาม 0  
  คำถามที่ค้าง: ['affected_people']
  คำขอพิกัด: ['ask_location:pending']
เบอร์ที่ยืนยันแล้ว: 0
คำขออนุญาตใช้เบอร์: ['pending']


## 2.1 แชร์ตำแหน่งและแยกการอนุญาตส่งต่อ (จำลอง LINE)

- แชร์ตอบคำขอที่ส่งสำเร็จแล้วและยังไม่หมดอายุ → บันทึก consent ของ incident นั้น
- แชร์มาก่อนเล่าเหตุ → ยังไม่สร้างเคส ถามก่อนว่าต้องการแจ้งอะไร
- เมื่อมีเหตุแล้วแต่พิกัดไม่ได้ตอบคำขอ → ขออนุญาตแนบพิกัดแยก
- ไม่อนุญาตส่งต่อ ≠ ขอลบข้อมูล
- ขั้นรับพิกัด/ปุ่มไม่เรียก Extraction; เคสทั่วไปอาจเรียบเรียงคำถามถัดไปตามแผน

พิกัดด้านล่างเป็นข้อมูลสมมติ ไม่ใช้ GPS เดาตำบลที่ยืนยันแล้ว


In [19]:
# ต่อจากส่วน 2: เคยได้รับคำขอพิกัดแล้ว แม้ให้จุดสังเกตไปก็ยังแชร์ตอบคำขอได้ภายใน 24 ชม.
output = agent(location=(14.9512, 100.3646))
print(output.output_text)
agent.show_state()


บันทึกข้อมูลไว้แล้ว กำลังส่งต่อค่ะ

ขอทราบว่ามีคนได้รับผลกระทบกี่คน หากไม่ทราบจำนวนบอกได้ค่ะ
incident inc_9b060e0284b9  status=submitted  matching=new  urgent=['help_requested', 'people_trapped']
  location_area          known       ถาม 0  {"raw": "หมู่ 3 ตำบลน้ำตาล", "stated_level": "unspecified", "match_status": "unverified", "candidates": []}
  location_landmark      known       ถาม 0  "หลังวัด"
  location_coordinates   known       ถาม 1  {"latitude": 14.9512, "longitude": 100.3646, "source": "line_location", "message_id": "msg_72b9077e9066"}
  situation              known       ถาม 0  {"text": "ช่วยด้วยค่ะ ยายติดอยู่บนหลังคา น้ำขึ้นเร็วมาก หมู่ 3 ตำบลน้ำตาล เบอร์ [เบอร์โทร]", "tags": ["water_rising"]}
  needs                  not_asked   ถาม 0  
  affected_people        not_asked   ถาม 2  
  vulnerable_people      known       ถาม 0  {"present": true, "groups": [{"type": "elderly", "count": 1, "is_approximate": false}]}
  water_level            not_asked   ถาม 0  
  contact         

In [20]:
# แชร์อย่างเดียว: ยังไม่มี incident และยังไม่มี consent
agent = make_agent()
output = agent(location=(14.9512, 100.3646))
print(output.output_text)
assert output.incident_id is None

# เล่าเหตุภายหลัง: นำตำแหน่งที่พักไว้มาใช้ แล้วขออนุญาตส่งต่อแยก
output = agent(input_text="น้ำท่วมบ้านค่ะ")
print(output.output_text)
print("ปุ่ม:", [(q.label, q.data) for q in output.quick_replies])


รับตำแหน่งแล้วค่ะ ต้องการแจ้งเหตุอะไรที่จุดนี้คะ กรุณาเล่าสถานการณ์ก่อนค่ะ

ระบบจะใช้ข้อมูลที่คุณแจ้งเพื่อจัดทำรายงานและส่งต่อให้ผู้ดูแลรายงานเพื่อประสานความช่วยเหลือค่ะ
รับทราบข้อมูลที่แจ้งค่ะ

เหตุเกิดที่หมู่บ้าน ตำบล หรืออำเภอใด กรุณาบอกชื่อพื้นที่ค่ะ

ขออนุญาตแนบตำแหน่งนี้ไปกับรายงาน ให้ผู้ดูแลรายงานใช้ประกอบการพิจารณาได้ไหมคะ หากไม่ต้องการ ตอบว่าไม่อนุญาตได้ค่ะ
ปุ่ม: [('อนุญาต', 'location_consent=yes'), ('ไม่อนุญาต', 'location_consent=no')]


In [21]:
# เลือกปุ่มจากคำตอบล่าสุด เพื่อให้ตรงกับคำขอที่กำลังรอ
button = next(q for q in output.quick_replies if q.data == "location_consent=no")
output = agent(postback=button.data)
print(output.output_text)
print("ยังมี incident:", agent.state is not None, "| ขอให้ลบข้อมูล:", agent.deletion_requested_at is not None)


รับทราบค่ะ จะไม่แนบตำแหน่งนี้ไปกับรายงาน หากสะดวก บอกจุดสังเกตหรือพื้นที่กว้างขึ้นแทนได้ค่ะ

ขอชื่อหมู่บ้าน ตำบล หรืออำเภอที่เกิดเหตุค่ะ
ยังมี incident: True | ขอให้ลบข้อมูล: False


หน้าตา message ที่จะส่งให้ LINE Reply API (มีปุ่ม Quick Reply ติดกับข้อความสุดท้าย)

In [22]:
from app.line_adapter import build_messages

preview_agent = make_agent()
# เป็นเพียง payload preview ไม่ส่งข้อความเข้า LINE จริง
build_messages(preview_agent(input_text="น้ำท่วมบ้านค่ะ"))


[{'type': 'text',
  'text': 'รับทราบข้อมูลที่แจ้งค่ะ\n\nจุดเกิดเหตุอยู่ที่ไหนคะ? แชร์ตำแหน่ง หรือพิมพ์หมู่บ้าน ตำบล และจุดสังเกตได้ค่ะ หากแจ้งแทนผู้อื่น กรุณาระบุตำแหน่งของจุดเกิดเหตุ ตำแหน่งจะใช้ระบุจุดเกิดเหตุและแนบรายงานให้ผู้ดูแลรายงานเพื่อประสานความช่วยเหลือ หากไม่สะดวก สามารถข้ามได้ค่ะ\n\nระบบจะใช้ข้อมูลที่คุณแจ้งเพื่อจัดทำรายงานและส่งต่อให้ผู้ดูแลรายงานเพื่อประสานความช่วยเหลือค่ะ',
  'quickReply': {'items': [{'type': 'action',
     'action': {'type': 'location', 'label': '📍 แชร์ตำแหน่ง'}},
    {'type': 'action',
     'action': {'type': 'postback',
      'label': 'ข้าม',
      'data': 'location_share=skip',
      'displayText': 'ข้าม'}}]}}]

## 3. ลองเอง

พิมพ์ข้อความที่อยากทดสอบ แล้วรันซ้ำได้เรื่อย ๆ ใช้ `agent.reset()` เมื่ออยากเริ่มเรื่องใหม่

In [23]:
output = agent(input_text="ต้องตัดไฟตอนไหนคะ")
print(output.output_text)
agent.show_last()

ขณะนี้ยังไม่มีข้อมูลยืนยันเพียงพอสำหรับตอบคำถามนี้ ขอแนะนำสอบถามหน่วยงานที่เกี่ยวข้องผ่านช่องทางต่อไปนี้ค่ะ
• สายด่วน ปภ. (ยังไม่ได้ตรวจ) 1784
• การแพทย์ฉุกเฉิน (ยังไม่ได้ตรวจ) 1669

ตอนนี้ต้องการความช่วยเหลือเรื่องใด หรือเพียงแจ้งให้ทราบ กรุณาบอกได้เลยค่ะ
👤 ต้องตัดไฟตอนไหนคะ
   ✦ intents=['ask_information'] target=active_incident
   ➜ actions=['ask_follow_up', 'answer_question']


## 4. ตรวจ event ที่ Agent 2 จะได้รับ

ตัวอย่างไฟล์แรกเป็น submission เวลา 18:40 version 1 และไฟล์ที่สองเป็น update เวลา 18:42 version 2
`incident_updated` ส่งทั้ง `changes` และ `report` ล่าสุด ตรวจ JSON ด้วย schema จริงก่อนส่งให้เพื่อน


In [24]:
from pydantic import TypeAdapter
from app.schemas import HandoffEvent

for filename in ("agent2_input_example.json", "agent2_update_example.json"):
    event = TypeAdapter(HandoffEvent).validate_json((APP_ROOT / filename).read_text(encoding="utf-8"))
    print(filename, event.event_type.value, "case_version =", event.case_version)


agent2_input_example.json incident_submitted case_version = 1
agent2_update_example.json incident_updated case_version = 2


In [25]:
# ดูสถานะ outbox โดยไม่แสดงข้อมูลส่วนบุคคล
[(record.event_type.value, record.event_id, record.delivery_status.value, record.attempts)
 for record in agent.repository.outbox(agent.internal_user_id)]


[]

## 5. ยืนยันอนุญาตใช้เบอร์แยกจากการยืนยันรายงาน

รันต่อจากเคสที่มีเบอร์รอยืนยัน และตอบคำถามจนถึงขั้น contact ก่อน
หากยังไม่มีปุ่ม consent รอบนี้ จะไม่กดแทนผู้ใช้หรือถือว่ายินยอมเอง


In [26]:
contact_button = next((q for q in output.quick_replies
                       if q.data and q.data.startswith("contact_consent=yes:")), None)
if contact_button:
    output = agent(postback=contact_button.data)  # จำลองผู้ใช้กดอนุญาตอย่างชัดเจน
    print(output.output_text)
else:
    print("ยังไม่มีคำขออนุญาตใช้เบอร์ที่แสดงในคำตอบล่าสุด")


ยังไม่มีคำขออนุญาตใช้เบอร์ที่แสดงในคำตอบล่าสุด


## 6. Live end-to-end: เรียก Gemini จริง โดยไม่ต่อ LINE

รันส่วน setup ด้านบนก่อน แล้วรันส่วนนี้ทีละเซลล์ได้เลย ไม่ต้องรันบทสนทนาตัวอย่างส่วน 1–5
ใช้ key ใน `dspy-agent/.env` หรือ `lab/1/.env` และใช้ข้อมูลเหตุการณ์สมมติเท่านั้น

- เรียก Gemini จริง ทั้ง Extraction และการเรียบเรียงในรอบที่จำเป็น
- Python Rules, state, consent, priority และ outbox ทำงานจริง
- ฐานข้อมูลจำลองเป็น SQLite ในหน่วยความจำ ไม่แตะข้อมูล API ที่เปิดอยู่
- ไม่ส่งเข้า LINE หรือ Agent 2: เห็น payload ที่เตรียมส่งและสถานะ `pending` ตามจริง
- ปิด cache สำหรับส่วนนี้ เพื่อดูการเรียกโมเดลใหม่ ใช้เวลาและมีค่าใช้จ่ายตามจริง


In [27]:
from time import perf_counter
from app.schemas import IncidentStatus, EventType, UrgentFlag, FieldStatus

# num_retries=0 ช่วยไม่ให้การทดสอบวนซ้ำเมื่อ key/โมเดลใช้ไม่ได้
lm = dspy.LM(model=model, api_key=api_key, cache=False, timeout=30, num_retries=0)
dspy.configure(lm=lm)


def live_turn(test_agent, message):
    started = perf_counter()
    before = len(lm.history)
    result = test_agent(input_text=message)
    elapsed = perf_counter() - started
    print(result.output_text)
    print("เวลา (วินาที):", round(elapsed, 2), "| รายการใน lm.history เพิ่ม:", len(lm.history) - before)
    print("สถานะ:", result.incident_status, "| การกระทำ:", [a.value for a in result.next_actions])
    print("ถามต่อ:", [f.value for f in result.pending_question.field_names] if result.pending_question else [])
    return result


In [28]:
# A. เหตุทั่วไป: state ต้องเก็บสิ่งที่ผู้ใช้บอก และถามจุดเกิดเหตุ/GPS ทางเลือก
live_general = make_agent()
result_general = live_turn(live_general, "น้ำท่วมเข้าบ้านค่ะ ยังไม่ได้แจ้งที่ไหน")
assert live_general.state is not None, "Extraction ยังไม่ได้ข้อมูลเหตุ"
assert live_general.state.status == IncidentStatus.DRAFT
assert live_general.state.situation.status == FieldStatus.KNOWN
print("PASS: เก็บ state ของเหตุทั่วไปได้")


รับทราบข้อมูลที่แจ้งค่ะ

จุดเกิดเหตุอยู่ที่ไหนคะ? แชร์ตำแหน่ง หรือพิมพ์หมู่บ้าน ตำบล และจุดสังเกตได้ค่ะ หากแจ้งแทนผู้อื่น กรุณาระบุตำแหน่งของจุดเกิดเหตุ ตำแหน่งจะใช้ระบุจุดเกิดเหตุและแนบรายงานให้ผู้ดูแลรายงานเพื่อประสานความช่วยเหลือ หากไม่สะดวก สามารถข้ามได้ค่ะ

ระบบจะใช้ข้อมูลที่คุณแจ้งเพื่อจัดทำรายงานและส่งต่อให้ผู้ดูแลรายงานเพื่อประสานความช่วยเหลือค่ะ
เวลา (วินาที): 2.77 | รายการใน lm.history เพิ่ม: 2
สถานะ: IncidentStatus.DRAFT | การกระทำ: ['ask_follow_up']
ถามต่อ: ['location_area', 'location_coordinates']
PASS: เก็บ state ของเหตุทั่วไปได้


In [29]:
# B. เหตุด่วน: ส่งรายงานลง outbox ทันที ไม่รอข้อมูลครบหรือยืนยัน
live_urgent = make_agent()
result_urgent = live_turn(live_urgent, "ช่วยด้วยค่ะ ยายติดอยู่บนหลังคา น้ำขึ้นเร็วมาก หมู่ 3 ตำบลน้ำตาล")
assert live_urgent.state.status == IncidentStatus.SUBMITTED
assert UrgentFlag.PEOPLE_TRAPPED in live_urgent.state.urgent_flag_types
assert live_urgent.state.priority.level.value == "red"
records = live_urgent.repository.outbox(live_urgent.internal_user_id)
assert records[0].event_type == EventType.INCIDENT_SUBMITTED
print("PASS: บันทึก submission จริง | delivery =", records[0].delivery_status.value)


จากข้อมูลที่แจ้ง กรุณาติดต่อช่องทางฉุกเฉินเพื่อขอความช่วยเหลือทันทีค่ะ
• สายด่วน ปภ. (ยังไม่ได้ตรวจ) 1784
• การแพทย์ฉุกเฉิน (ยังไม่ได้ตรวจ) 1669

บันทึกข้อมูลไว้แล้ว กำลังส่งต่อค่ะ

จุดเกิดเหตุอยู่ที่ไหนคะ? แชร์ตำแหน่ง หรือพิมพ์หมู่บ้าน ตำบล และจุดสังเกตได้ค่ะ หากแจ้งแทนผู้อื่น กรุณาระบุตำแหน่งของจุดเกิดเหตุ ตำแหน่งจะใช้ระบุจุดเกิดเหตุและแนบรายงานให้ผู้ดูแลรายงานเพื่อประสานความช่วยเหลือ หากไม่สะดวก สามารถข้ามได้ค่ะ

ระบบจะใช้ข้อมูลที่คุณแจ้งเพื่อจัดทำรายงานและส่งต่อให้ผู้ดูแลรายงานเพื่อประสานความช่วยเหลือค่ะ
เวลา (วินาที): 1.39 | รายการใน lm.history เพิ่ม: 1
สถานะ: IncidentStatus.SUBMITTED | การกระทำ: ['submit_report', 'ask_follow_up']
ถามต่อ: ['location_coordinates']
PASS: บันทึก submission จริง | delivery = pending


In [30]:
# C. อัปเดตข้อมูล: ใช้ incident เดิม เพิ่ม version และส่ง report ฉบับล่าสุดคู่กับ changes
incident_id_before = live_urgent.state.incident_id
result_update = live_turn(live_urgent, "มีผู้ได้รับผลกระทบ 2 คนค่ะ เป็นผู้สูงอายุ 1 คน")
assert live_urgent.state.incident_id == incident_id_before
records = live_urgent.repository.outbox(live_urgent.internal_user_id)
updates = [r for r in records if r.event_type == EventType.INCIDENT_UPDATED]
assert updates, "ยังไม่มี update — ดู agent.show_last() เพื่อดูว่าโมเดลดึงข้อมูลอะไรได้"
print("PASS: incident เดิม | case_version =", updates[-1].payload["case_version"])
print(json.dumps(updates[-1].payload, ensure_ascii=False, indent=2))


บันทึกข้อมูลไว้แล้ว กำลังส่งต่อค่ะ

ขอจุดสังเกตใกล้ที่เกิดเหตุ เช่น วัด โรงเรียน หรือชื่อถนนค่ะ
เวลา (วินาที): 1.09 | รายการใน lm.history เพิ่ม: 1
สถานะ: IncidentStatus.SUBMITTED | การกระทำ: ['send_update', 'ask_follow_up']
ถามต่อ: ['location_landmark']
PASS: incident เดิม | case_version = 2
{
  "event_id": "evt_4bf1f9f733d7",
  "schema_version": "1.3",
  "occurred_at": "2026-11-01T19:41:39.840040+07:00",
  "case_version": 2,
  "event_type": "incident_updated",
  "incident_id": "inc_d1584663b5b2",
  "incident_version": 2,
  "changes": [
    {
      "field": "affected_people",
      "change_type": "initial",
      "previous_value": null,
      "previous_reported_at": null,
      "new_value": {
        "status": "known",
        "value": {
          "present": true,
          "count": 2,
          "is_approximate": false
        }
      },
      "reported_at": "2026-11-01T19:41:39.840040+07:00"
    }
  ],
  "report": {
    "incident_id": "inc_d1584663b5b2",
    "incident_version": 2,
   

### D. ดูว่า LLM เรียบเรียงเองได้จริงหรือถูก validator ปฏิเสธ

เรียกขั้นเรียบเรียงโดยตรงเพื่อดูคำตอบดิบและผลตรวจแยกกัน คำตอบดิบในส่วนนี้ยังไม่ใช่คำตอบที่จะส่งผู้ใช้
หาก `validate_wording` ไม่ผ่าน ตัว Agent จะเลือก template สำรอง ไม่ใช่ถือว่าคำตอบดิบผ่านแล้ว


In [31]:
from app.response import ComposeReply, validate_wording

reply_plan = {
    "message": "น้ำเข้าบ้านแล้วค่ะ ตอนนี้กังวลมาก",
    "recent_messages": [],
    "confirmed_state": {
        "situation": {"status": "known", "value": {"text": "น้ำเข้าบ้าน", "tags": ["water_in_house"]}},
        "needs": {"status": "not_asked", "value": None},
    },
    "changes": [],
    "question_hints": {"needs": ["ตอนนี้ต้องการความช่วยเหลือเรื่องใดค่ะ"]},
    "acknowledgement_hints": ["รับทราบข้อมูลที่แจ้งค่ะ"],
    "actions": ["ask_follow_up"],
    "evidence": [],
    "max_chars": 1200,
}
started = perf_counter()
with dspy.context(adapter=dspy.ChatAdapter(use_json_adapter_fallback=False)):
    raw_wording = dspy.Predict(ComposeReply)(plan=json.dumps(reply_plan, ensure_ascii=False)).result
print(raw_wording.model_dump_json(indent=2))
print("ผ่าน validator:", validate_wording(raw_wording, reply_plan))
print("เวลา (วินาที):", round(perf_counter() - started, 2))


{
  "acknowledgement": "ได้รับทราบข้อมูลสถานการณ์น้ำที่เข้าบ้านแล้วนะคะ",
  "questions": {
    "needs": "ในตอนนี้ทางคุณต้องการให้ช่วยเหลือด้านใดเป็นพิเศษไหมคะ"
  },
  "knowledge_answer": "",
  "source_ids": []
}
ผ่าน validator: False
เวลา (วินาที): 1.36


### ดูผลที่รันโดยผู้ช่วยไว้ล่าสุด (ถ้ามี)

ไฟล์นี้เป็นหลักฐานจากการเรียกโมเดลจริงครั้งนั้น ใช้แยกจากผลทดสอบแบบจำลองและไม่รับประกันว่ารันครั้งใหม่จะได้คำตอบเหมือนเดิม


In [32]:
results_path = ROOT / "lab" / "1" / "live_llm_results.json"
if results_path.exists():
    print(results_path.read_text(encoding="utf-8"))
else:
    print("ยังไม่มีผลรัน Gemini จริงที่บันทึกไว้ — รันส่วน A–D ด้านบนเพื่อทดลอง")


{
  "run_at": "2026-11-01T19:39:05.008268+07:00",
  "model": "gemini-3.1-flash-lite",
  "real_llm": true,
  "line_connected": false,
  "agent2_connected": false,
  "tests": [
    {
      "name": "A_general_intake",
      "passed": true,
      "checks_failed": [],
      "seconds": 2.643,
      "lm_history_entries": 2,
      "output_text": "รับทราบข้อมูลที่แจ้งค่ะ\n\nจุดเกิดเหตุอยู่ที่ไหนคะ? แชร์ตำแหน่ง หรือพิมพ์หมู่บ้าน ตำบล และจุดสังเกตได้ค่ะ หากแจ้งแทนผู้อื่น กรุณาระบุตำแหน่งของจุดเกิดเหตุ ตำแหน่งจะใช้ระบุจุดเกิดเหตุและแนบรายงานให้ผู้ดูแลรายงานเพื่อประสานความช่วยเหลือ หากไม่สะดวก สามารถข้ามได้ค่ะ\n\nระบบจะใช้ข้อมูลที่คุณแจ้งเพื่อจัดทำรายงานและส่งต่อให้ผู้ดูแลรายงานเพื่อประสานความช่วยเหลือค่ะ",
      "incident_status": "draft",
      "next_actions": [
        "ask_follow_up"
      ],
      "pending_fields": [
        "location_area",
        "location_coordinates"
      ],
      "priority": {
        "level": "green",
        "reasons": [
          "water_in_house"
        ],
        "